基于[原 repo ](https://github.com/xianglin226/DSSC)所述的依赖版本创建环境后（完整环境见 [dssc_origin.yml](../.env/dssc_origin.yml)），进行如下检查：

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 1.6.0
PyTorch CUDA: 10.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


↑ CUDA available: True 但是并不能说明这个 PyTorch 真能在 4090 上计算，CUDA 10.2 不支持 4090，需要使用 CUDA 11+ 的较新 PyTorch binary

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))

print("\nSupported architectures:")
print(torch.cuda.get_arch_list())

print("\nTrying actual GPU computation...")
x = torch.randn(1000, 1000, device="cuda")
y = x @ x
print(y.device)
print("GPU computation succeeded.")

PyTorch: 1.6.0
CUDA runtime: 10.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Capability: (8, 9)

Supported architectures:
['sm_37', 'sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'compute_37']

Trying actual GPU computation...


↑ 计算卡住，现在这套 PyTorch 能通过驱动识别 4090，却没有适用于 4090 的 CUDA kernel

为了使 DSSC 正常运行，暂时不采用原 repo 依赖，新环境的测试及各测试阶段 yml 见 .archive 目录

↓ [指定使用 cuda 11](../.env/dssc.yml) 创建环境，新环境创建成功（先前使用 pip 下载 dgl 时出现错误，故本次并未在 yml 中指定 dgl，待其他依赖解决后再解决 dgl 问题），测试：

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))

print("\nSupported architectures:")
print(torch.cuda.get_arch_list())

print("\nTrying actual GPU computation...")
x = torch.randn(1000, 1000, device="cuda")
y = x @ x
print(y.device)
print("GPU computation succeeded.")

PyTorch: 2.1.2
CUDA runtime: 11.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Capability: (8, 9)

Supported architectures:
['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_37', 'sm_90', 'compute_37']

Trying actual GPU computation...
cuda:0
GPU computation succeeded.


↑ cuda 11 可用，现在尝试下载 dgl

下载 dgl 1.1.3 + cuda 11.8，保留 --no-deps，防止 pip 修改已经验证正常的 pytorch 和 cuda 版本

In [2]:
%%bash

python -m pip install "dgl==1.1.3+cu118" \
  -f https://data.dgl.ai/wheels/cu118/repo.html \
  --no-deps

Looking in links: https://data.dgl.ai/wheels/cu118/repo.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 MB 18.3 MB/s eta 0:00:0000:0100:01


测试 dgl 是否可用：

In [3]:
import torch
import dgl
from dgl.nn import GATConv

print("=" * 50)
print("DGL / PyTorch / GPU Test")
print("=" * 50)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("DGL:", dgl.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Construct a small graph directly on the GPU
src = torch.tensor([0, 1, 2, 3, 0, 2], device="cuda")
dst = torch.tensor([1, 2, 3, 0, 2, 0], device="cuda")

g = dgl.graph(
    (src, dst),
    num_nodes=4,
    device="cuda"
)

print("\nGraph:")
print(g)
print("Graph device:", g.device)

# Node features
x = torch.randn(
    4,
    16,
    device="cuda"
)

# DSSC uses GATConv
gat = GATConv(
    in_feats=16,
    out_feats=8,
    num_heads=2,
    allow_zero_in_degree=True
).cuda()

print("\nRunning GATConv...")

y = gat(g, x)

torch.cuda.synchronize()

print("Output shape:", y.shape)
print("Output device:", y.device)

print("\nDGL GATConv GPU test PASSED!")

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
DGL / PyTorch / GPU Test
PyTorch: 2.1.2
PyTorch CUDA: 11.8
DGL: 1.1.3+cu118
GPU: NVIDIA GeForce RTX 4090

Graph:
Graph(num_nodes=4, num_edges=6,
      ndata_schemes={}
      edata_schemes={})
Graph device: cuda:0

Running GATConv...
Output shape: torch.Size([4, 2, 8])
Output device: cuda:0

DGL GATConv GPU test PASSED!


↑ 可行，完整环境见：[dssc_4090](../.env/dssc_4090.yml)